# Session 3 — Community Detection: Finding Modules Automatically

**Goal of this session:** replace "I can see two blocks in the heatmap" with an algorithm, and see exactly how much your choice of one parameter changes what it finds.

*Network Neuroscience in Python, session 3 of 10.*

## Why this matters

Eyeballing a correlation matrix for blocks works for a 9-region toy example. It does not work for a 200-region parcellation, and it is not reproducible even when it does work — two people staring at the same heatmap will draw the boundary in different places. **Community detection** algorithms find groups of densely-interconnected nodes automatically, from the graph structure alone. The catch, and the actual point of this session, is that "automatically" does not mean "objectively" — these algorithms take a parameter, and that parameter is a choice you're making, not a fact the data hands you.

## The toy network, again

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def generate_toy_network(n_per_module=5, n_modules=2, p_within=0.7, p_between=0.05,
                          add_connector=True, connector_frac=0.6, seed=0):
    """Same function as session 1. Not a brain."""
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    node_id = 0
    modules = []
    for m in range(n_modules):
        members = []
        for _ in range(n_per_module):
            G.add_node(node_id, module=m)
            members.append(node_id)
            node_id += 1
        modules.append(members)
    for m in range(n_modules):
        members = modules[m]
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if rng.random() < p_within:
                    G.add_edge(members[i], members[j])
    for m1 in range(n_modules):
        for m2 in range(m1 + 1, n_modules):
            for u in modules[m1]:
                for v in modules[m2]:
                    if rng.random() < p_between:
                        G.add_edge(u, v)
    if add_connector:
        connector = node_id
        G.add_node(connector, module="connector")
        n_link = max(1, round(connector_frac * n_per_module))
        for members in modules:
            chosen = rng.choice(members, size=min(n_link, len(members)), replace=False)
            for other in chosen:
                G.add_edge(connector, int(other))
    return G


def draw_toy_network(G, ax=None, title="", node_colours=None):
    if node_colours is None:
        palette = {0: "#2b6cb0", 1: "#dd6b20", 2: "#805ad5", "connector": "#38a169"}
        node_colours = [palette[G.nodes[n]["module"]] for n in G.nodes()]
    node_sizes = [500 + 220 * G.degree(n) for n in G.nodes()]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))
    pos = nx.spring_layout(G, seed=1)
    nx.draw_networkx_edges(G, pos, alpha=0.4, width=1.6, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=node_colours, node_size=node_sizes,
                            edgecolors="white", linewidths=1.2, alpha=0.95, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=11, font_color="white",
                             font_weight="bold", ax=ax)
    ax.set_title(title, fontsize=13)
    ax.axis("off")
    return pos


G = generate_toy_network(seed=0)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

## Modularity: what the algorithm is actually optimising

Almost every community-detection method in this space is, underneath, searching for a partition of nodes into groups that maximises a score called **modularity, Q**. Modularity compares the fraction of edges that fall *inside* the proposed groups to how many you'd expect inside those same groups if edges were placed at random (respecting each node's degree — the configuration-model idea from session 1, again).

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

You don't need to hand-compute this — `networkx` does it — but the idea matters: **Q is high when there are more within-group edges than a degree-matched random graph would produce.** A partition that puts every node in its own group scores Q = 0 by construction (no edges are "within" a singleton). A partition that puts every node in one giant group also tends to score low, because it credits the group with edges a random graph would have produced anyway. The useful partitions live in between.

**Higher Q is not automatically "more correct".** It tells you a partition explains the wiring well relative to chance — it says nothing about whether that partition matches any real, meaningful boundary in the system you're studying.

## Louvain community detection

The **Louvain algorithm** is the standard fast, greedy way to (approximately) maximise modularity: it repeatedly moves individual nodes into whichever neighbouring group increases Q the most, then merges the resulting groups into "super-nodes" and repeats. `networkx` ships it as `nx.community.louvain_communities`.

It takes a **resolution parameter** (default 1.0). Turning resolution *down* makes the algorithm prefer fewer, larger communities; turning it *up* makes it prefer more, smaller ones. There is no principled "correct" resolution built into the method — it is a knob you turn, and different values are appropriate for different questions.

In [ ]:
communities_default = nx.community.louvain_communities(G, resolution=1.0, seed=0)
Q_default = nx.community.modularity(G, communities_default)

print(f"resolution=1.0 -> {len(communities_default)} communities, Q={Q_default:.3f}")
for i, comm in enumerate(communities_default):
    print(f"  community {i}: {sorted(comm)}")

## The same network, five resolutions

Nothing about the underlying graph changes between these runs. Only the resolution parameter does.

In [ ]:
resolutions = [0.3, 0.5, 1.0, 1.6, 2.5]
sweep = []
for res in resolutions:
    comms = nx.community.louvain_communities(G, resolution=res, seed=0)
    q = nx.community.modularity(G, comms)
    sweep.append((res, comms, q))
    print(f"resolution={res:.1f}   n_communities={len(comms)}   Q={q:.3f}   "
          f"sizes={sorted((len(c) for c in comms), reverse=True)}")

## Watching the partition change

Five small networks, same layout, same node positions, coloured only by whichever partition that resolution produced. Watch the palette reshuffle as resolution increases from left to right.

In [ ]:
fig, axes = plt.subplots(1, len(sweep), figsize=(4 * len(sweep), 4.2))
palette = ["#2b6cb0", "#dd6b20", "#805ad5", "#38a169", "#e53e3e", "#d69e2e"]
pos = nx.spring_layout(G, seed=1)

for ax, (res, comms, q) in zip(axes, sweep):
    node_to_colour = {}
    for i, comm in enumerate(comms):
        for n in comm:
            node_to_colour[n] = palette[i % len(palette)]
    colours = [node_to_colour[n] for n in G.nodes()]
    nx.draw_networkx_edges(G, pos, alpha=0.35, width=1.3, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=colours, node_size=380,
                            edgecolors="white", linewidths=1, ax=ax)
    ax.set_title(f"resolution={res}\n{len(comms)} communities, Q={q:.2f}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Communities and modularity, as resolution sweeps continuously

The panel above jumps between five hand-picked resolutions. Here's the full picture: community count and modularity Q, plotted against a continuous sweep of resolution values. Notice Q does not simply keep climbing as the partition fragments — it peaks somewhere in the middle and falls off in both directions.

In [ ]:
res_fine = np.linspace(0.1, 3.0, 30)
n_comms_fine, q_fine = [], []
for res in res_fine:
    comms = nx.community.louvain_communities(G, resolution=res, seed=0)
    n_comms_fine.append(len(comms))
    q_fine.append(nx.community.modularity(G, comms))

fig, ax1 = plt.subplots(figsize=(9, 5.5))
ax1.plot(res_fine, n_comms_fine, "o-", color="#2b6cb0", label="number of communities")
ax1.set_xlabel("resolution parameter", fontsize=12)
ax1.set_ylabel("number of communities", color="#2b6cb0", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#2b6cb0")

ax2 = ax1.twinx()
ax2.plot(res_fine, q_fine, "s-", color="#c53030", label="modularity Q")
ax2.set_ylabel("modularity Q", color="#c53030", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#c53030")

ax1.set_title("Community count and modularity both depend on a parameter you chose", fontsize=13)
plt.tight_layout()
plt.show()

## The actual point

At resolution 0.3 the algorithm sees one big community. At resolution 2.5 it sees five small ones. Both of those are legitimate outputs of a legitimate algorithm run on the exact same, unchanged network. "How many communities does this brain network have" is therefore not a question with a single correct answer — it is a question that only makes sense alongside a stated resolution, the same way "how many peaks does this signal have" only makes sense alongside a stated smoothing bandwidth.

That does not make community detection useless. It means: report your resolution, show that your finding is stable across a reasonable range of it (exactly the kind of sweep we just plotted), and be suspicious of any paper that reports a community count without one.

**Next session:** now that we can label nodes by community, we can finally define what a "hub" really means — a node with high degree that is also either bridging communities or anchoring one.